# TalkingTrees shared functions
Act notebooks run these definitions during setup. Edit the data source, display labels, or plotting functions here.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://amazon-talking-trees.github.io/talking-trees/data/Weather%2BTranspiration_HalfHourly.csv"
PANEL_LABELS = {
    "Rain": "Rain (monthly total)",
    "Light": "Light (daytime mean)",
    "Temperature": "Temperature (daytime mean)",
    "Humidity": "Humidity (daytime mean)",
}
AXIS_UNITS = {
    "Rain": "mm/month",
    "Light": "W/m²",
    "Temperature": "°C",
    "Humidity": "% Relative Humidity",
}
YEAR_COLORS = {"Normal": "cornflowerblue", "Drought": "firebrick"}

In [ ]:
def load_data(include_diurnals=False):
    columns = ["date", "year", "month", "hour", "YearType", *PANEL_LABELS]
    if include_diurnals:
        columns += ["Evap.Power", "Tr.Erisma", "Tr.Itauba"]
    climate = pd.read_csv(DATA_URL, usecols=columns)
    print(f"Loaded {len(climate):,} half-hourly observations.")
    return climate[columns]

In [ ]:
def get_monthly_summary(data):
    groups = ["year", "month", "YearType"]
    rain = data.groupby(groups)["Rain"].sum(min_count=1)

    # Average the other weather measurements from 09:30 through 16:30.
    daytime = data.loc[(data["hour"] > 9) & (data["hour"] < 17)]
    means = daytime.groupby(groups)[["Light", "Temperature", "Humidity"]].mean()
    return pd.concat([rain, means], axis=1).reset_index()

In [ ]:
def plot_monthly_data(data, variables, years):
    if not isinstance(variables, list) or not variables or any(
        name not in PANEL_LABELS for name in variables
    ):
        raise ValueError('Choose a list of column names, such as ["Rain", "Humidity"]. '
                         'Available names: Rain, Light, Temperature, Humidity.')
    if not isinstance(years, list) or not years or any(year not in YEAR_COLORS for year in years):
        raise ValueError('Use ["Normal"], ["Drought"], or ["Normal", "Drought"] for years.')

    fig, panels = plt.subplots(len(variables), 1, sharex=True, squeeze=False,
                               figsize=(7, 1.9 * len(variables) + 1.1), layout="constrained")
    for ax, variable in zip(panels[:, 0], variables):
        ax.axvspan(0.5, 6.5, color="#F2F2F2")
        ax.axvspan(6.5, 12.5, color="#FFF8DC")
        for year_type in years:
            subset = data.loc[data["YearType"] == year_type].sort_values("month")
            ax.plot(subset["month"], subset[variable], color=YEAR_COLORS[year_type],
                    marker="o", markersize=4, label=year_type)
        ax.set_title(PANEL_LABELS[variable], loc="left", fontsize=11)
        ax.set_ylabel(AXIS_UNITS[variable])
        ax.set_xlim(0.5, 12.5)
        ax.spines[["top", "right"]].set_visible(False)

    panels[0, 0].legend(loc="best", ncol=len(years))
    panels[-1, 0].set_xticks([2, 4, 6, 8, 10, 12])
    panels[-1, 0].set_xlabel("Month")
    fig.suptitle("Amazon weather · " + " and ".join(years) + "\nWet season: Jan–Jun    |    Dry season: Jul–Dec",
                 fontsize=12)
    plt.show()

## Act 2 · Diurnal curves
Sap flow is used as a proxy for transpiration. The source table has no units for the two `Tr.` columns. Update the sap-flow axis label once the conversion and reference area are confirmed.

In [1]:
DIURNAL_LABELS = {
    "Light": "Light\n(W/m²)",
    "Temperature": "Temperature\n(°C)",
    "Humidity": "Humidity\n(%)",
    "Evap.Power": "Evaporative\npower",
    "Rain": "Rain per\nhalf-hour",
}


def graph_diurnals(data, vars, n_days=6):
    """Plot daytime weather and transpiration on randomly sampled normal-year dates."""
    if not isinstance(vars, list) or not vars or any(v not in DIURNAL_LABELS for v in vars):
        raise ValueError(f"Choose one or more variables from {list(DIURNAL_LABELS)}.")
    if type(n_days) is not int or n_days < 1:
        raise ValueError("Set n_days to a positive whole number.")

    daytime = data.loc[data["YearType"].eq("Normal") & data["hour"].between(5, 20)]
    dates = daytime["date"].dropna().drop_duplicates()
    if n_days > len(dates):
        raise ValueError(f"Choose at most {len(dates)} days.")
    sampled = dates.sample(n=n_days).sort_values(
        key=lambda dates: pd.to_datetime(dates, format="%m/%d/%y")
    )

    fig, axes = plt.subplots(
        len(vars) + 1, n_days,
        squeeze=False, sharex=True, sharey="row",
        figsize=(2.7 + 2.2 * n_days, 1.5 * (len(vars) + 1) + 1.1),
        layout="constrained",
    )

    hours = [i / 2 for i in range(10, 41)]
    for col, date in enumerate(sampled):
        day = daytime.loc[daytime["date"].eq(date)].set_index("hour").reindex(hours)
        axes[0, col].set_title(str(date))

        for row, variable in enumerate(vars):
            ax = axes[row, col]
            ax.plot(hours, day[variable], color="cornflowerblue")
            if col == 0:
                ax.set_ylabel(DIURNAL_LABELS[variable])

        ax = axes[-1, col]
        ax.plot(hours, day["Tr.Erisma"], label="Erisma", color="#0072B2")
        ax.plot(hours, day["Tr.Itauba"], label="Itauba", color="#E69F00")
        ax.set_xlim(5, 20)
        ax.set_xticks([6, 12, 18])

    transpiration_ax = axes[-1, 0]
    transpiration_ax.set_ylabel("Transpiration\n(mm/hour)")
    transpiration_ax.set_ylim(bottom=min(0, transpiration_ax.get_ylim()[0]))

    # Reserve space on the right for the legend and below the panels for the x-axis title.
    handles, labels = transpiration_ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside right lower", frameon=False)
    fig.supxlabel("Hour of day")
    fig.suptitle("Weather and tree transpiration during a normal year")
    plt.show()